# Smoke test - single-config end-to-end pipeline check

Set `config_f_name` in the next cell to any `generate_minimize` config.
The notebook derives the scope, fold and result paths from the config
itself, so the rest of the cells adapt automatically. Verifies:

1. The evaluation completes and writes `results_<fold>_<run>.json` with
   all expected aggregated metrics (GED, FED, OC, Correctness, Runtime,
   etc.).
2. The new per-instance JSON dump (Note A) writes one
   `cf_<instance_id>.json` per test instance, each carrying the input
   graph, the counterfactual, and the per-instance metrics - exactly
   what E2 (plausibility) and E4/E5 (per-instance diagnostics) need to
   run as post-processing.

If anything is missing, this notebook fails loudly before launching the
full batch.

## 1. Paths and imports

In [1]:
import sys
import os
import re
import json
from pathlib import Path

# Make the repo importable when this notebook runs from lab/
module_path = os.path.abspath(os.path.join('..'))
sys.path.insert(0, module_path)
os.chdir(module_path)

from src.utils.context import Context
from src.evaluation.future.evaluator_manager_triplets import EvaluatorManager
from src.data_analysis.future.data_analyzer import DataAnalyzer

# ============================================================
# CHANGE THIS to point at any generate_minimize config.
# Scope, fold and result paths are derived from the file itself.
# ============================================================
config_f_name = 'lab/config/generate_minimize/asd/rsgg/rsgg-lcls-seed0/generate_minimize0.jsonc'
runno = 1

config_path = os.path.join(module_path, config_f_name)
assert os.path.isfile(config_path), f"config not found: {config_path}"

def _read_jsonc(path):
    text = Path(path).read_text()
    text = re.sub(r'/\*.*?\*/', '', text, flags=re.DOTALL)
    text = re.sub(r'//.*?\n', '\n', text)
    return json.loads(text)

# Derive scope from the config and fold from the filename.
_cfg = _read_jsonc(config_path)
scope = _cfg['experiment']['scope']
_m = re.search(r'generate_minimize(\d+)\.jsonc$', config_f_name)
assert _m, f"could not parse fold id out of {config_f_name}"
fold = int(_m.group(1))
results_root = Path('lab/output/results') / scope

print(f"config_f_name : {config_f_name}")
print(f"scope         : {scope}")
print(f"fold          : {fold}")
print(f"runno         : {runno}")
print(f"results root  : {results_root}/...")

config_f_name : lab/config/generate_minimize/asd/rsgg/rsgg-lcls-seed0/generate_minimize0.jsonc
scope         : asd_rsgg_lcls_seed0
fold          : 0
runno         : 1
results root  : lab/output/results/asd_rsgg_lcls_seed0/...


## 2. Run the evaluation

Builds the context, loads the dataset and oracle (cached if present),
instantiates DCEM + LocalSearch and processes the entire test set of
fold 0. If DCEM medoids are not cached this will train them first
(roughly minutes for Synthie).

In [2]:
print(f"Generating context for: {config_path}")
context = Context.get_context(config_path)
context.run_number = runno

context.logger.info(f"Executing: {context.config_file} Run: {context.run_number}")
context.logger.info("Creating the evaluation manager ...")
eval_manager = EvaluatorManager(context)

context.logger.info("Evaluating ...")
eval_manager.evaluate()
print("EVALUATION FINISHED")

Generating context for: /home/rodrigo/projects/GRETEL stuff/GRETEL/lab/config/generate_minimize/asd/rsgg/rsgg-lcls-seed0/generate_minimize0.jsonc
2026-06-10 02:16:36,349 | INFO | 173128 - Executing: /home/rodrigo/projects/GRETEL stuff/GRETEL/lab/config/generate_minimize/asd/rsgg/rsgg-lcls-seed0/generate_minimize0.jsonc Run: 1
2026-06-10 02:16:36,362 | INFO | 173128 - Creating the evaluation manager ...
2026-06-10 02:16:36,395 | INFO | 173128 - Loading: ASD-f562e3436929ceb8a1635f7742d2ab12
2026-06-10 02:16:36,398 | INFO | 173128 - Instantiating: src.dataset.generators.asd.ASD
2026-06-10 02:16:36,566 | INFO | 173128 - Apply: src.dataset.manipulators.padding.AdjacencyMatrixPadder
2026-06-10 02:16:36,567 | INFO | 173128 - Instantiating: src.dataset.manipulators.padding.AdjacencyMatrixPadder
2026-06-10 02:16:36,591 | INFO | 173128 - Apply: src.dataset.manipulators.centralities.NodeCentrality
2026-06-10 02:16:36,592 | INFO | 173128 - Instantiating: src.dataset.manipulators.centralities.NodeC

KeyboardInterrupt: 

## 3. Verify aggregated results file

Confirm `results_<fold>_<runno>.json` was written and contains entries
for every stage in the minimizing pipeline (Runtime, GED, FED,
Correctness, OC, OracleAccuracy, Sparsity).

In [ ]:
results_files = list(results_root.rglob(f'results_{fold}_{runno}.json'))
assert results_files, f"no results_{fold}_{runno}.json under {results_root}"
results_path = results_files[0]
print(f"results file: {results_path}")

agg = json.loads(results_path.read_text())
stages = list(agg.get('results', {}).keys())
print(f"stages recorded: {len(stages)}")
for s in stages:
    short = s.rsplit('.', 1)[-1]
    n_entries = len(agg['results'][s])
    sample = agg['results'][s][:1]
    print(f"  {short:<22} n={n_entries:<5} sample={sample}")

expected = {'Runtime', 'GraphEditDistance', 'FeatureEditDistance',
            'Correctness', 'OracleCalls', 'OracleAccuracy', 'Sparsity'}
present = {s.rsplit('.', 1)[-1] for s in stages}
missing = expected - present
print(f"\nexpected stages missing: {missing if missing else 'NONE'}")

## 4. Verify per-instance JSON dumps (Note A)

For each instance the pipeline should have written
`cf_<instance_id>.json` to
`<scope>/<dataset>/<oracle>/<explainer>/cf_per_instance/fold_<fold>/`
containing `input`, `counterfactual`, and `metrics`.

In [ ]:
dump_dirs = list(results_root.rglob(f'cf_per_instance/fold_{fold}'))
assert dump_dirs, f"no cf_per_instance/fold_{fold} dir under {results_root}"
dump_dir = dump_dirs[0]
files = sorted(dump_dir.glob('cf_*.json'))
print(f"per-instance dump dir: {dump_dir}")
print(f"cf_*.json files: {len(files)}")
assert len(files) > 0, "expected at least one per-instance dump"

# Inspect the first one - keys, shapes, metrics presence.
payload = json.loads(files[0].read_text())
print(f"\nfirst file: {files[0].name}")
print(f"  top-level keys: {sorted(payload.keys())}")

required_top = {'id', 'fold_id', 'input', 'counterfactual', 'metrics'}
assert required_top.issubset(payload.keys()), f"missing keys {required_top - set(payload.keys())}"

print(f"  input keys:           {sorted(payload['input'].keys())}")
if payload['counterfactual'] is not None:
    print(f"  counterfactual keys:  {sorted(payload['counterfactual'].keys())}")
    print(f"  input edges count:    {len(payload['input']['edges'])}")
    print(f"  cf edges count:       {len(payload['counterfactual']['edges'])}")
else:
    print('  counterfactual: null  (generator failed to produce a CF on this instance)')

print(f"  metrics keys:         {sorted(payload['metrics'].keys())}")
print(f"  metrics:              {payload['metrics']}")

## 5. Quick aggregated view via DataAnalyzer

Sanity-check that the new scope shows up in the project's aggregation
pipeline (the same code that produces the thesis tables/figures).

In [ ]:
df = DataAnalyzer.create_aggregated_dataframe('lab/output/results')
row = df[df['scope'] == scope]
assert not row.empty, f"scope '{scope}' missing in aggregated dataframe"
display_cols = ['scope', 'GraphEditDistance', 'GraphEditDistance-std',
                'FeatureEditDistance', 'OracleCalls', 'Correctness', 'Runtime']
row[display_cols]

## What 'pass' looks like

* Section 3 prints all 7 expected stages with non-empty entries.
* Section 4 prints at least one `cf_<id>.json` whose payload has
  `input`, `counterfactual` and `metrics` keys; `metrics` contains the
  same 7 names.
* Section 5 returns a single row whose `scope` matches the one printed
  in section 1, with numeric values for GED / FED / OC / Correctness /
  Runtime.

If any assertion fails, the corresponding part of the pipeline is
broken and the full batch should NOT be launched yet.